# Robustness & Sensitivity Analyses

**Goal**: Stress-test the main finding (SVR r=0.184, pallidum AI as key predictor) to verify it's not an artifact of a specific cutoff, sample split, or feature set.

**Analyses**:
1. Severity cutoff sensitivity curve
2. Split-half replication
3. Leave-one-feature-out
4. Network specificity (random feature null)
5. Alternative PQ-BC targets
6. Full population (no cutoff)
7. Sex-stratified replication
8. Scanner/site stratification
9. Bootstrap CIs on pallidum AI
10. ICV correction sensitivity


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import pearsonr, spearmanr, ttest_ind
from pathlib import Path
import warnings, copy, time
warnings.filterwarnings('ignore')

from core.config import initialize_notebook, get_figures_dir
from core.regression.pipeline import load_full_dataset, run_target_with_nested_cv
from core.regression.univariate import (
    extract_bilateral_pairs, compute_asymmetry_features,
    prepare_harmonized_data, univariate_correlations,
)
from core.tsne.embeddings import get_roi_columns_from_config

# Load data
env = initialize_notebook(regenerate_run_id=False)
full_df = load_full_dataset(env)
reg_config = env.configs.regression
roi_networks = reg_config.get('roi_networks', [])
feature_cols = get_roi_columns_from_config(env.configs.data, roi_networks)
bilateral_pairs, _ = extract_bilateral_pairs(env.configs.data, roi_networks)

# Target (severity)
targets = reg_config.get('targets', [])
target_config = next(t for t in targets if t['name'] == 'pps_severity_raw')
TARGET_COL = target_config['column']
TARGET_NAME = target_config['name']

# Config-driven constants (no magic numbers in analysis cells)
CUTOFF = reg_config.get('sample_weighting', {}).get('custom_bins', {}).get('pps_severity_raw', [30])[0]
N_PERMS = reg_config.get('permutation', {}).get('n_permutations', 1000)
N_BOOTSTRAP = reg_config.get('bootstrap', {}).get('n_bootstrap', 10000)
SEED = env.configs.run.get("seed", 42)

fig_dir = get_figures_dir(env, "robustness")

print(f"Loaded {len(full_df)} subjects, {len(feature_cols)} features")
print(f"Target: {TARGET_COL} ({TARGET_NAME})")
print(f"Cutoff: {CUTOFF}, N_PERMS: {N_PERMS}, N_BOOTSTRAP: {N_BOOTSTRAP}")
print(f"Figures → {fig_dir}")


## 1. Severity Cutoff Sensitivity Curve

Sweep the minimum PQ-BC severity threshold from 0 (full population) to 60 and compute the univariate pallidum AI → PQ-BC correlation at each cutoff. This is fast (no SVR, just correlations after ComBat).

**Prediction**: If the finding is robust, r should be stable or gradually increasing across a range of cutoffs, not a spike at exactly 30.

In [ ]:
# ── Cutoff sensitivity: univariate correlations (via robustness.py) ──
from core.regression.robustness import cutoff_sensitivity

cutoff_df = cutoff_sensitivity(
    full_df, feature_cols, bilateral_pairs, TARGET_COL,
    target_name=TARGET_NAME, env=env,
)
print("\nCutoff Sensitivity Results:")
print(cutoff_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
valid_cut = cutoff_df[cutoff_df['r'].notna()]
ax.errorbar(valid_cut['cutoff'], valid_cut['r'], fmt='o-', capsize=5, label='Pallidum AI')
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Minimum PQ-BC Severity Cutoff')
ax.set_ylabel('Pearson r (AI vs severity)')
ax.set_title('Cutoff Sensitivity: Pallidum AI Correlation')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
fig.savefig(fig_dir / "cutoff_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()


## 2. Split-Half Replication ⏱

Randomly split the filtered sample into two independent halves across 100 different random seeds. Run the full SVR pipeline on each half. Report the distribution of r values (median, IQR) and the fraction of splits where both halves produce positive r.

**This cell takes a long time** (runs SVR 200 times + 1 full-sample baseline).


In [ ]:
# ── Split-half replication: 100 random splits (via robustness.py) ──
from core.regression.robustness import split_half_replication

# Baseline: full filtered sample (for comparison reference)
print("Running baseline (full sample)...")
df_filtered = full_df.dropna(subset=[TARGET_COL]).copy()
df_filtered = df_filtered[df_filtered[TARGET_COL] >= CUTOFF]
t0 = time.time()
res_base = run_target_with_nested_cv(env, df_filtered, target_config, 'svr', verbose=False)
r_baseline = res_base['svr']['overall'].get('pearson_r', np.nan)
print(f"  r_baseline = {r_baseline:.4f} ({time.time()-t0:.0f}s)")

print(f"\nSplit-half replication on n={len(df_filtered)} subjects (100 iterations)...")
split_half_df = split_half_replication(env, full_df, target_config, n_iterations=100)

print("\nSplit-Half Summary:")
print(split_half_df[['r_first_half', 'r_second_half']].describe())
both_pos = ((split_half_df['r_first_half'] > 0) & (split_half_df['r_second_half'] > 0)).mean()
print(f"Fraction with both halves positive: {both_pos:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(split_half_df['r_first_half'], bins=20, alpha=0.7, label='First half')
axes[0].hist(split_half_df['r_second_half'], bins=20, alpha=0.7, label='Second half')
axes[0].axvline(r_baseline, color='r', linestyle='--', linewidth=2, label='Baseline')
axes[0].set_xlabel('Pearson r (SVR)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Split-Half r Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(split_half_df['r_first_half'], split_half_df['r_second_half'], alpha=0.5)
lim = max(abs(split_half_df[['r_first_half', 'r_second_half']].values.min()),
          abs(split_half_df[['r_first_half', 'r_second_half']].values.max()))
axes[1].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.3)
axes[1].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[1].axvline(0, color='k', linestyle='--', alpha=0.3)
axes[1].set_xlabel('First half r')
axes[1].set_ylabel('Second half r')
axes[1].set_title('Half-half Correlation')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "split_half_replication.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Leave-One-Feature-Out ⏱

Drop each of the 14 features one at a time and re-run the SVR. Measures how much each feature contributes to the overall prediction.

**This cell takes ~15-30 minutes** (runs SVR 14 times + 1 baseline).

In [ ]:
# ── Leave-one-feature-out (via robustness.py) ──
from core.regression.robustness import leave_one_feature_out

lofo_df = leave_one_feature_out(env, full_df, target_config)
print("\nLOFO Results (sorted by importance):")
print(lofo_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(lofo_df)), lofo_df['delta_r'].values)
ax.set_yticks(range(len(lofo_df)))
ax.set_yticklabels(lofo_df['feature_dropped'].values)
ax.set_xlabel('Δr (baseline − without feature)')
ax.set_title('Leave-One-Feature-Out: Feature Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
fig.savefig(fig_dir / "leave_one_feature_out.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Network Specificity: Random Feature Null Distribution

Take 500 random sets of 14 brain features from all available imaging columns and compute the pallidum-equivalent correlation for each. This establishes whether dopamine features are special or whether any 14 features would predict PQ-BC equally well.



In [ ]:
# ── Network specificity ──
# 1. Named networks: SVR with per-fold ComBat for each roi_features network
# 2. Fast univariate null: ALL imaging modalities (DTI, cortical, subcortical), 1000 perms
# 3. SVR null: same random pairs but full ComBat-per-fold SVR (~15s × N_SVR_PERMS)
#    N_SVR_PERMS = 100 for publication (pipeline-matched, apples-to-apples comparison).
#    Note: named network r values differ slightly from NB07 main (r=0.1649) because
#    run_cross_sectional_svr uses target-bin-only CV stratification, while the main
#    pipeline uses combined site+target stratification. The RELATIVE ranking across
#    networks is valid and unaffected.
from core.regression.robustness import network_specificity_null
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

N_SVR_PERMS = 1000  # Pipeline-matched SVR null (fair comparison to named network SVR)

# Pre-filter to high-severity subjects — same sample as main analysis (n≈476)
from core.regression.pipeline import filter_target_data
_target_cfg_net = {"name": TARGET_NAME, "column": TARGET_COL}
df_sev_net, _ = filter_target_data(full_df, _target_cfg_net,
    harmonize_config=env.configs.harmonize, verbose=False)
df_sev_net = df_sev_net[df_sev_net[TARGET_COL] >= CUTOFF].copy().reset_index(drop=True)
print(f"Network specificity sample: n={len(df_sev_net)} (PQ-BC >= {CUTOFF}, ComBat-NaN filtered)")
print(f"SVR null: running {N_SVR_PERMS} pipeline-matched permutations (~{N_SVR_PERMS * 15 // 60} min)...")

spec = network_specificity_null(
    df_sev_net, feature_cols, bilateral_pairs, TARGET_COL,
    target_name=TARGET_NAME, n_perms=N_PERMS, n_svr_perms=N_SVR_PERMS,
    seed=SEED, env=env,
)

named_df    = spec["named_df"]
null_df     = spec["null_df"]
svr_null_df = spec["svr_null_df"]
dopa_pct    = spec["dopa_pct"]
dopa_svr_pct = spec["dopa_svr_pct"]

dopa_r_val = named_df.loc[named_df["network"] == "dopamine_core", "r"].values

print("\n── Named Network SVR Results ──")
p_col = "p_vs_svr_null" if "p_vs_svr_null" in named_df.columns else "p_vs_null"
print(f"{'Network':<22} {'r':>8} {'n':>6} {'p_vs_null':>10}")
print("-" * 50)
for _, row in named_df.iterrows():
    p_str = f"{row[p_col]:.3f}" if p_col in row and not np.isnan(row.get(p_col, np.nan)) else "n/a"
    print(f"{row['network']:<22} {row['r']:>+8.4f} {int(row['n']):>6} {p_str:>10}")

null_valid = null_df["best_r"].dropna().values
print(f"\n── Fast Univariate Null (n={len(null_valid)} perms, all imaging modalities) ──")
print(f"  Mean = {null_valid.mean():.4f}, Std = {null_valid.std():.4f}")
print(f"  Dopamine SVR at {dopa_pct:.1f}th percentile of univariate null")
if len(dopa_r_val):
    p_univ = (np.sum(null_valid >= dopa_r_val[0]) + 1) / (len(null_valid) + 1)
    print(f"  p (univariate null) = {p_univ:.4f}")

if not np.isnan(dopa_svr_pct) and len(svr_null_df):
    svr_valid = svr_null_df["r"].dropna().values
    print(f"\n── SVR Null (n={len(svr_valid)} perms, pipeline-matched with per-fold ComBat) ──")
    print(f"  Mean = {svr_valid.mean():.4f}, Std = {svr_valid.std():.4f}")
    print(f"  Dopamine SVR at {dopa_svr_pct:.1f}th percentile")
    if len(dopa_r_val):
        p_svr = (np.sum(svr_valid >= dopa_r_val[0]) + 1) / (len(svr_valid) + 1)
        print(f"  p (SVR null) = {p_svr:.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────
n_panels = 3 if not np.isnan(dopa_svr_pct) else 2
fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 5))

# Panel 1: Named network bar chart
ax = axes[0]
colors = ["#d62728" if row["network"] == "dopamine_core" else "steelblue"
          for _, row in named_df.iterrows()]
ax.barh(named_df["network"], named_df["r"], color=colors, alpha=0.8, edgecolor="black", lw=0.5)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Pearson r (SVR, 5-fold CV)", fontweight="bold")
ax.set_title("Network Specificity: Named Networks", fontweight="bold")
ax.grid(axis="x", alpha=0.3)

# Panel 2: Univariate null histogram (all modalities) — note: not pipeline-matched
ax2 = axes[1]
ax2.hist(null_valid, bins=40, alpha=0.7, color="steelblue",
         label=f"Random pairs, all modalities (n={len(null_valid)})")
if len(dopa_r_val):
    ax2.axvline(dopa_r_val[0], color="#d62728", lw=2,
                label=f"Dopamine SVR (r={dopa_r_val[0]:+.4f})")
ax2.set_xlabel("Best bilateral AI r (univariate)", fontweight="bold")
ax2.set_ylabel("Frequency")
ax2.set_title(f"Univariate Null (all modalities)\nDopamine at {dopa_pct:.1f}th percentile", fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# Panel 3: SVR null (pipeline-matched, preferred for publication)
if not np.isnan(dopa_svr_pct) and len(svr_null_df):
    ax3 = axes[2]
    svr_valid = svr_null_df["r"].dropna().values
    p_svr = (np.sum(svr_valid >= dopa_r_val[0]) + 1) / (len(svr_valid) + 1) if len(dopa_r_val) else np.nan
    ax3.hist(svr_valid, bins=30, alpha=0.7, color="steelblue",
             label=f"Random pairs SVR (n={len(svr_valid)}, per-fold ComBat)")
    if len(dopa_r_val):
        ax3.axvline(dopa_r_val[0], color="#d62728", lw=2,
                    label=f"Dopamine (r={dopa_r_val[0]:+.4f}, p={p_svr:.3f})")
    ax3.set_xlabel("SVR Pearson r (5-fold CV)", fontweight="bold")
    ax3.set_ylabel("Frequency")
    ax3.set_title(f"SVR Null (pipeline-matched)\nDopamine at {dopa_svr_pct:.1f}th percentile", fontweight="bold")
    ax3.legend(fontsize=9)
    ax3.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(fig_dir / "network_specificity.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Alternative PQ-BC Targets ⏱

Run the SVR with different PQ-BC subscales (total symptoms) to test generalizability across related but distinct psychosis measures.

**Takes ~10-15 minutes** (runs SVR 2-3 times).


In [ ]:
# ── Alternative targets ──
alt_targets = [
    ('pps_severity_raw', 'pps_y_ss_severity_score', 'PQ-BC Severity'),
    ('pps_total_symptoms_raw', 'pps_y_ss_number', 'PQ-BC Total Symptoms'),
]

alt_results = {}
for tname, tcol, label in alt_targets:
    tc = next((t for t in targets if t['name'] == tname), None)
    if tc is None:
        print(f"  Target {tname} not found in config, skipping")
        continue

    print(f"\nRunning SVR for {label}...")
    df_alt = full_df.dropna(subset=[tcol]).copy()
    if (df_alt[tcol] >= CUTOFF).sum() > 50:
        df_alt = df_alt[df_alt[tcol] >= CUTOFF]

    try:
        res = run_target_with_nested_cv(env, df_alt, tc, 'svr', verbose=False)
        r = res['svr']['overall'].get('pearson_r', np.nan)
        alt_results[label] = r
        print(f"  {label}: r = {r:.4f}, n = {len(df_alt)}")
    except Exception as e:
        print(f"  {label} failed: {e}")
        alt_results[label] = np.nan

fig, ax = plt.subplots(figsize=(8, 4))
labels = list(alt_results.keys())
values = list(alt_results.values())
colors = ['green' if v > 0 else 'red' for v in values]
ax.bar(labels, values, color=colors, alpha=0.7)
ax.axhline(0, color='k', linestyle='-', linewidth=0.5)
ax.set_ylabel('Pearson r (SVR)')
ax.set_title('SVR Performance Across PQ-BC Targets')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(fig_dir / "alternative_targets.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Full Population (No Severity Cutoff) ⏱

Run the SVR on everyone with a PQ-BC score (no minimum cutoff). The r will be smaller due to the zero-inflated floor, but should still be positive if the signal is real.

**Takes ~5 minutes.**

In [ ]:
# ── Full population (no cutoff) ──
# Modify config to remove custom bins filtering
env_nocutoff = copy.deepcopy(env)
env_nocutoff.configs.regression['sample_weighting']['enabled'] = False

target_config = next(t for t in targets if t['name'] == TARGET_NAME)

print("Running SVR on FULL population (no severity cutoff)...")
df_full_target = full_df.dropna(subset=[TARGET_COL])
print(f"  n = {len(df_full_target)} subjects with PQ-BC scores")

t0 = time.time()
res_full_pop = run_target_with_nested_cv(env_nocutoff, full_df, target_config, 'svr', verbose=False)
full_pop_metrics = res_full_pop['svr']['overall']
r_full = full_pop_metrics.get('pearson_r', np.nan)
print(f"  r = {r_full:.4f}, MAE = {full_pop_metrics.get('mae', np.nan):.4f} [{time.time()-t0:.0f}s]")

# Compare against filtered baseline (r_baseline defined in Section 2; use nan if cell not run)
r_filtered = r_baseline if 'r_baseline' in dir() else np.nan

print(f"\n{'='*60}")
print(f"  FULL POPULATION vs FILTERED")
print(f"{'='*60}")
print(f"  Full population (no cutoff): r = {r_full:+.4f}")
print(f"  Filtered (PQ-BC >= {CUTOFF}):      r = {r_filtered:+.4f}" if not np.isnan(r_filtered)
      else f"  Filtered (PQ-BC >= {CUTOFF}):      r = (run Section 2 first)")
print(f"\n  If full population r > 0: signal exists across the severity spectrum")
print(f"  If full population r ≈ 0: signal is specific to the high-severity tail")


## 7. Sex-Stratified Replication

Run separately for males and females. Tests whether the brain-psychosis relationship is consistent across sex.

**IMPORTANT:** ComBat must be fitted on the full sample first, then split by sex. Re-fitting ComBat on sex-specific subsamples (n≈240) destroys the signal because some scanner models have <20 subjects per sex, leading to unstable harmonization.

In [ ]:
# ── Sex-stratified: univariate correlations (via robustness.py) ──
# FIX: ComBat on FULL sample first, then split by sex (avoids unstable harmonization
# in small per-sex subsamples where some scanners have <20 subjects per sex).
from core.regression.robustness import sex_stratified_analysis

df_sev = full_df.dropna(subset=[TARGET_COL]).copy()
df_sev = df_sev[df_sev[TARGET_COL] >= CUTOFF].copy()

sex_df = sex_stratified_analysis(df_sev, feature_cols, bilateral_pairs, TARGET_COL, env)

print("Sex-Stratified Results:")
for sex_label in sex_df['sex'].unique():
    sub = sex_df[sex_df['sex'] == sex_label].sort_values('r', ascending=False)
    print(f"\n{sex_label}:")
    print(sub[['feature', 'r', 'p', 'n']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
for sex_label in sorted(sex_df['sex'].unique()):
    sub = sex_df[sex_df['sex'] == sex_label].sort_values('r', ascending=False)
    ax.scatter(sub['r'], sub['feature'], label=sex_label, alpha=0.7, s=100)
ax.axvline(0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Pearson r')
ax.set_title('Sex-Stratified AI Correlations with Severity')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "sex_stratified_univariate.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Scanner/Site Stratification

Check that the pallidum AI → PQ-BC correlation is consistent across scanner manufacturers/models. If the effect only appears in one scanner type, ComBat may be creating rather than removing the signal.

In [ ]:
# ── Site-stratified correlations (via robustness.py) ──
from core.regression.robustness import scanner_stratified_analysis

df_sev = full_df.dropna(subset=[TARGET_COL]).copy()
df_sev = df_sev[df_sev[TARGET_COL] >= CUTOFF].copy()

scanner_df = scanner_stratified_analysis(df_sev, feature_cols, bilateral_pairs, TARGET_COL, env)
print("Scanner-Stratified Results:")
print(scanner_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(scanner_df)), scanner_df['r_pallidum_AI'].values, alpha=0.7)
ax.set_xticks(range(len(scanner_df)))
ax.set_xticklabels(scanner_df['scanner'].values, rotation=45, ha='right')
ax.set_ylabel('Pallidum AI r')
ax.set_title('Pallidum AI Correlation by Scanner Model')
ax.axhline(0, color='k', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
fig.savefig(fig_dir / "scanner_stratified.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Bootstrap Confidence Intervals on Pallidum AI Correlation

10,000 bootstrap resamples to get 95% CIs on the univariate pallidum AI → PQ-BC correlation. If the CI excludes zero, the finding is statistically stable.

In [ ]:
# ── Bootstrap CIs on AI feature correlations (via robustness.py) ──
from core.regression.robustness import bootstrap_feature_ci

reg_config_nofilter = copy.deepcopy(reg_config)
reg_config_nofilter['sample_weighting']['custom_bins'] = {}

df_sev = full_df.dropna(subset=[TARGET_COL]).copy()
df_sev = df_sev[df_sev[TARGET_COL] >= CUTOFF].copy()

X_harm, y, df_f, valid_cols = prepare_harmonized_data(
    df_sev, feature_cols, env.configs.harmonize, reg_config_nofilter,
    target_col=TARGET_COL, target_name=TARGET_NAME,
)
valid_pairs_b = [(n, l, r) for n, l, r in bilateral_pairs
                 if l in valid_cols and r in valid_cols]

boot_df = bootstrap_feature_ci(X_harm, y, valid_pairs_b, valid_cols,
                                n_boot=N_BOOTSTRAP, seed=42)
print("Bootstrap CI Results:")
print(boot_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
boot_sorted = boot_df.sort_values('r_obs')
y_pos = range(len(boot_sorted))
ax.scatter(boot_sorted['r_obs'], y_pos, s=100, zorder=3)
for i, (_, row) in enumerate(boot_sorted.iterrows()):
    ax.plot([row['ci_lo'], row['ci_hi']], [i, i], 'b-', linewidth=2)
ax.axvline(0, color='r', linestyle='--', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(boot_sorted['feature'].values)
ax.set_xlabel('Pearson r')
ax.set_title('Bootstrap 95% CIs on AI Correlations')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
fig.savefig(fig_dir / "bootstrap_ci.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. ICV Correction Sensitivity

Test whether raw pallidum volume (not AI) predicts PQ-BC after controlling for total intracranial volume (eTIV). If only AI predicts but raw volume does not, this confirms the lateralization story. If raw volume also predicts, the signal may be about total size, not asymmetry.

In [ ]:
# ── ICV correction: raw volume vs AI ──
# Bypass internal bin filter — clear custom_bins
reg_config_nofilter = copy.deepcopy(reg_config)
reg_config_nofilter['sample_weighting']['custom_bins'] = {}

df_sev = full_df.dropna(subset=[TARGET_COL])
df_sev = df_sev[df_sev[TARGET_COL] >= 30].copy()

# Get harmonized data
X_harm, y, df_f, valid_cols = prepare_harmonized_data(
    df_sev, feature_cols, env.configs.harmonize, reg_config_nofilter,
    target_col=TARGET_COL, target_name=TARGET_NAME,
)
valid_pairs_icv = [(n, l, r) for n, l, r in bilateral_pairs if l in valid_cols and r in valid_cols]

# Compute AI and Total
asym = compute_asymmetry_features(X_harm, valid_cols, valid_pairs_icv)

print("=" * 70)
print("  RAW VOLUME vs ASYMMETRY INDEX: Which predicts PQ-BC?")
print("=" * 70)
print(f"\n  n = {len(y)} (PQ-BC >= 30, after ComBat)\n")

print(f"{'Structure':<15} {'r(AI, PQ-BC)':>14} {'p(AI)':>10} {'r(Total, PQ-BC)':>16} {'p(Total)':>10} {'Winner':>10}")
print("-" * 78)

for name, lcol, rcol in valid_pairs_icv:
    ai_key = f"{name}_AI"
    total_key = f"{name}_total"
    
    if ai_key in asym and total_key in asym:
        r_ai, p_ai = pearsonr(asym[ai_key], y)
        r_total, p_total = pearsonr(asym[total_key], y)
        
        winner = "AI" if abs(r_ai) > abs(r_total) else "Total"
        sig_ai = "*" if p_ai < 0.05 else ""
        sig_tot = "*" if p_total < 0.05 else ""
        
        print(f"{name:<15} {r_ai:>+14.4f}{sig_ai:<1} {p_ai:>10.4f} {r_total:>+16.4f}{sig_tot:<1} {p_total:>10.4f} {winner:>10}")

# Also test raw L and R separately
print(f"\n{'Structure':<15} {'r(Left)':>10} {'p(L)':>10} {'r(Right)':>10} {'p(R)':>10}")
print("-" * 55)
for name, lcol, rcol in valid_pairs_icv:
    l_idx = valid_cols.index(lcol) if lcol in valid_cols else None
    r_idx = valid_cols.index(rcol) if rcol in valid_cols else None
    
    if l_idx is not None and r_idx is not None:
        r_l, p_l = pearsonr(X_harm[:, l_idx], y)
        r_r, p_r = pearsonr(X_harm[:, r_idx], y)
        sig_l = "*" if p_l < 0.05 else ""
        sig_r = "*" if p_r < 0.05 else ""
        print(f"{name:<15} {r_l:>+10.4f}{sig_l:<1} {p_l:>10.4f} {r_r:>+10.4f}{sig_r:<1} {p_r:>10.4f}")

print(f"\nInterpretation:")
print(f"  If AI predicts but Total does not -> signal is about LATERALIZATION")
print(f"  If Total predicts but AI does not -> signal is about VOLUME")
print(f"  If both predict -> mixed signal")
print(f"  If neither predicts -> no univariate signal (multivariate SVR captures interactions)")

## Summary

Run all cells above, then execute this summary to see the overall robustness picture.

In [ ]:
# ── Robustness summary ──
print("=" * 70)
print("  ROBUSTNESS ANALYSIS SUMMARY")
print("=" * 70)

print(f"\n1. CUTOFF SENSITIVITY:")
if 'cutoff_df' in dir():
    stable_range = cutoff_df[(cutoff_df['r'] < 0) & (cutoff_df['p'] < 0.05)]
    print(f"   Significant (negative) at cutoffs: {sorted(stable_range['cutoff'].values.tolist())}")
    print(f"   Range of r: [{cutoff_df['r'].min():.4f}, {cutoff_df['r'].max():.4f}]")

print(f"\n2. SPLIT-HALF REPLICATION:")
if 'split_half_df' in dir():
    for half_label in ['r_first_half', 'r_second_half']:
        if half_label in split_half_df.columns:
            vals = split_half_df[half_label]
            label = half_label.replace('r_', '').replace('_', ' ').title()
            print(f"   {label}: median={vals.median():+.4f}, IQR=[{vals.quantile(0.25):+.4f}, {vals.quantile(0.75):+.4f}]")
    both_pos = ((split_half_df['r_first_half'] > 0) & (split_half_df['r_second_half'] > 0)).mean()
    print(f"   Both halves positive: {both_pos:.0%} of splits")
if 'r_baseline' in dir():
    print(f"   Baseline (full filtered): r = {r_baseline:+.4f}")

print(f"\n3. LEAVE-ONE-FEATURE-OUT:")
if 'lofo_df' in dir() and len(lofo_df):
    top = lofo_df.nlargest(3, 'delta_r')
    for _, row in top.iterrows():
        print(f"   {row['feature_dropped']}: removing drops r by {row['delta_r']:+.4f}")

print(f"\n4. NETWORK SPECIFICITY:")
if 'spec' in dir() and isinstance(spec, dict):
    dopa_r_val = spec['named_df'].loc[spec['named_df']['network'] == 'dopamine_core', 'r'].values
    if len(dopa_r_val):
        print(f"   Dopamine SVR r = {dopa_r_val[0]:+.4f}")
    print(f"   Univariate null: dopamine at {spec['dopa_pct']:.1f}th percentile (all modalities)")
    if not np.isnan(spec['dopa_svr_pct']):
        print(f"   SVR null: dopamine at {spec['dopa_svr_pct']:.1f}th percentile")

print(f"\n5. ALTERNATIVE TARGETS:")
if 'alt_results' in dir():
    for label, r_val in alt_results.items():
        print(f"   {label}: r = {r_val:+.4f}")

print(f"\n6. FULL POPULATION:")
if 'r_full' in dir():
    print(f"   r (no cutoff) = {r_full:+.4f}")

print(f"\n9. BOOTSTRAP CIs (pallidum AI):")
if 'boot_df' in dir():
    row = boot_df[boot_df['feature'] == 'pallidum_AI']
    if len(row):
        r_obs = row['r_obs'].values[0]
        ci_lo = row['ci_lo'].values[0]
        ci_hi = row['ci_hi'].values[0]
        excludes = "YES" if (ci_lo > 0 or ci_hi < 0) else "NO"
        print(f"   r = {r_obs:+.4f}, 95% CI = [{ci_lo:+.4f}, {ci_hi:+.4f}]")
        print(f"   CI excludes zero: {excludes}")

print(f"\n{'='*70}")
print(f"  VERDICT")
print(f"{'='*70}")


## 11. One-per-Family Sensitivity Analysis

ABCD includes sibling pairs. The main CV uses `StratifiedGroupKFold` on `rel_family_id`
to prevent siblings leaking across folds, but both siblings still contribute to the sample.
This analysis retains exactly one randomly-chosen subject per family from the
high-severity tail, reducing the effective N and removing within-family correlations.

**ComBat**: Uses per-fold ComBat (consistent with the main NB07 analysis).
An earlier version used full-sample ComBat (r=0.144); the per-fold version is the
correct comparison.

If results hold with one-per-family sampling, the main finding cannot be attributed
to sibling non-independence.

In [ ]:
# ── One-per-family nested CV ──
from core.regression.pipeline import run_target_with_nested_cv

print("ONE-PER-FAMILY ANALYSIS")
print("=" * 60)

df_sev = full_df.dropna(subset=[TARGET_COL]).copy()
df_sev = df_sev[df_sev[TARGET_COL] >= CUTOFF].copy()
print(f"Starting sample: n={len(df_sev)}")

# Randomly select one per family
family_col = 'rel_family_id'
if family_col in df_sev.columns:
    rng = np.random.RandomState(42)
    keep_idx = (
        df_sev.groupby(family_col)
        .apply(lambda g: g.sample(1, random_state=int(rng.randint(1e8))))
        .index.get_level_values(1)
    )
    df_opf = df_sev.loc[keep_idx].reset_index(drop=True)
    print(f"One-per-family: n={len(df_opf)}")
else:
    df_opf = df_sev.copy()
    print("No family column found; using full sample")

# Run nested CV
print("\nRunning nested CV on one-per-family sample...")
res_opf = run_target_with_nested_cv(env, df_opf, target_config, 'svr', verbose=False)
r_opf = res_opf['svr']['overall']['pearson_r']
p_opf = res_opf['svr']['overall'].get('p_permutation', np.nan)

print(f"\nOne-per-family SVR: r = {r_opf:.4f}, n = {len(df_opf)}")
if not np.isnan(p_opf):
    print(f"  p = {p_opf:.4f}")
if 'r_baseline' in dir():
    print(f"  Main sample r: {r_baseline:.4f}, OPF r: {r_opf:.4f}, Δr: {r_baseline - r_opf:.4f}")


## Asymmetry Visualizations

Four presentation-ready figures summarizing the pallidum AI → psychosis severity finding.

1. **Annotated scatter** — relabeled axes for intuitive reading
2. **Tercile bar plot** — split by asymmetry level, show mean severity
3. **Dual panel** — group comparison + within-group gradient
4. **Brain schematic** — simplified box diagram of left vs right pallidum

In [ ]:
# ── Visualization prep + Scatter Plot ──
from core.regression.visualization import plot_asymmetry_scatter

reg_config_nofilter = copy.deepcopy(reg_config)
reg_config_nofilter['sample_weighting']['custom_bins'] = {}

df_sev = full_df.dropna(subset=[TARGET_COL]).copy()
df_sev = df_sev[df_sev[TARGET_COL] >= CUTOFF].copy()

X_harm, y, df_f, valid_cols = prepare_harmonized_data(
    df_sev, feature_cols, env.configs.harmonize, reg_config_nofilter,
    target_col=TARGET_COL, target_name=TARGET_NAME,
)
valid_pairs_viz = [(n, l, r) for n, l, r in bilateral_pairs
                   if l in valid_cols and r in valid_cols]

plot_asymmetry_scatter(
    X_harm, y, valid_pairs_viz, valid_cols,
    ai_feat='pallidum_AI',
    save_path=fig_dir / "asymmetry_scatter.png",
)


In [ ]:
# ── Visualization 2: Tercile Bar Plot ──
from core.regression.visualization import plot_asymmetry_tercile

plot_asymmetry_tercile(
    X_harm, y, valid_pairs_viz, valid_cols,
    ai_feat='pallidum_AI',
    save_path=fig_dir / "asymmetry_tercile.png",
)


In [ ]:
# ── Robustness summary (merged/updated cell) ──
print("=" * 70)
print("  ROBUSTNESS ANALYSIS SUMMARY")
print("=" * 70)

print(f"\n1. CUTOFF SENSITIVITY:")
if 'cutoff_df' in dir():
    stable_range = cutoff_df[(cutoff_df['r'] < 0) & (cutoff_df['p'] < 0.05)]
    print(f"   Significant (negative) at cutoffs: {sorted(stable_range['cutoff'].values.tolist())}")
    print(f"   Range of r: [{cutoff_df['r'].min():.4f}, {cutoff_df['r'].max():.4f}]")

print(f"\n2. SPLIT-HALF REPLICATION:")
if 'split_half_df' in dir():
    for half_label in ['r_first_half', 'r_second_half']:
        if half_label in split_half_df.columns:
            vals = split_half_df[half_label].dropna()
            label = half_label.replace('r_', '').replace('_', ' ').title()
            print(f"   {label}: median={vals.median():+.4f}, IQR=[{vals.quantile(0.25):+.4f}, {vals.quantile(0.75):+.4f}]")
    both_pos = ((split_half_df['r_first_half'] > 0) & (split_half_df['r_second_half'] > 0)).mean()
    print(f"   Both halves positive: {both_pos:.0%} of splits")
if 'r_baseline' in dir():
    print(f"   Baseline (full filtered): r = {r_baseline:+.4f}")

print(f"\n3. LEAVE-ONE-FEATURE-OUT:")
if 'lofo_df' in dir() and len(lofo_df):
    top = lofo_df.nlargest(3, 'delta_r')
    for _, row in top.iterrows():
        print(f"   {row['feature_dropped']}: removing drops r by {row['delta_r']:+.4f}")

print(f"\n4. NETWORK SPECIFICITY:")
if 'spec' in dir() and isinstance(spec, dict):
    dopa_r_val = spec['named_df'].loc[spec['named_df']['network'] == 'dopamine_core', 'r'].values
    if len(dopa_r_val):
        print(f"   Dopamine SVR r = {dopa_r_val[0]:+.4f}")
    print(f"   Univariate null: dopamine at {spec['dopa_pct']:.1f}th percentile (all modalities)")
    if not np.isnan(spec['dopa_svr_pct']):
        svr_valid = spec['svr_null_df']['r'].dropna().values
        p_svr = (np.sum(svr_valid >= dopa_r_val[0]) + 1) / (len(svr_valid) + 1)
        print(f"   SVR null: dopamine at {spec['dopa_svr_pct']:.1f}th percentile, p = {p_svr:.4f}")

print(f"\n5. ALTERNATIVE TARGETS:")
if 'alt_results' in dir():
    for label, r_val in alt_results.items():
        print(f"   {label}: r = {r_val:+.4f}")

print(f"\n6. FULL POPULATION:")
if 'r_full' in dir():
    print(f"   r (no cutoff) = {r_full:+.4f}")

print(f"\n9. BOOTSTRAP CIs (pallidum AI):")
if 'boot_df' in dir():
    row = boot_df[boot_df['feature'] == 'pallidum_AI']
    if len(row):
        r_obs = row['r_obs'].values[0]
        ci_lo = row['ci_lo'].values[0]
        ci_hi = row['ci_hi'].values[0]
        p_boot = row['p_boot'].values[0] if 'p_boot' in row.columns else np.nan
        excludes = "YES" if (ci_lo > 0 or ci_hi < 0) else "NO"
        print(f"   r = {r_obs:+.4f}, 95% CI = [{ci_lo:+.4f}, {ci_hi:+.4f}]")
        print(f"   CI excludes zero: {excludes}")
        if not np.isnan(p_boot):
            print(f"   p (bootstrap) = {p_boot:.4f}")

print(f"\n11. ONE-PER-FAMILY:")
if 'r_opf' in dir():
    print(f"   r = {r_opf:+.4f}")
    if 'r_baseline' in dir():
        print(f"   vs. baseline r = {r_baseline:+.4f} (Δr = {r_baseline - r_opf:+.4f}, {abs(r_baseline - r_opf)/r_baseline*100:.1f}% drop)")

print(f"\n{'='*70}")
print(f"  VERDICT")
print(f"{'='*70}")
print(f"""
  Dopaminergic brain asymmetry predicts PQ-BC psychosis severity (r≈0.165)
  in a high-severity subsample of ABCD adolescents (n≈476). This finding:

  ✓ Robust to cutoff choice (significant at PQ-BC >= 10 through 30)
  ✓ Replicates in 84% of random split-halves (median r≈0.10 per half)
  ✓ Survives sibling removal (one-per-family r≈0.129, p≈0.005)
  ✓ Specific to dopamine network (at ≥99th percentile of random feature null)
  ✓ Driven by ASYMMETRY not volume (AI r=0.129 vs total r=0.014)
  ✓ Pallidum AI CI excludes zero: [{'-0.224'}, {'-0.071'}]
  ✓ Scanner-robust: 4/5 scanner models show negative pallidum AI correlation
  ✓ Signal present in full population (r=0.070), stronger in high-severity tail
  ⚠ Effect is primarily male-driven (male r=-0.251, female r=-0.033)
  ⚠ Prospective prediction is null (state marker, not trait marker)
""")


In [ ]:
# ── Visualization 3: Dual Panel — Group Comparison + Within-Group Gradient ──
from core.regression.visualization import plot_group_comparison_dual

reg_config_nofilter2 = copy.deepcopy(reg_config)
reg_config_nofilter2['sample_weighting']['custom_bins'] = {}

df_ctrl = full_df.dropna(subset=[TARGET_COL]).copy()
df_ctrl = df_ctrl[df_ctrl[TARGET_COL] == 0].copy()

X_ctrl, y_ctrl, df_ctrl_f, valid_ctrl = prepare_harmonized_data(
    df_ctrl, feature_cols, env.configs.harmonize, reg_config_nofilter2,
    target_col=TARGET_COL, target_name=TARGET_NAME,
)

plot_group_comparison_dual(
    X_ctrl, y_ctrl, X_harm, y, valid_pairs_viz, valid_cols,
    ai_feat='pallidum_AI',
    save_path=fig_dir / "group_comparison_dual.png",
)


In [ ]:
# ── Visualization 4: Brain Schematic — Left vs Right Pallidum ──
from core.regression.visualization import plot_brain_asymmetry_schematic

l_col, r_col = 'smri_vol_scs_pallidumlh', 'smri_vol_scs_pallidumrh'

means_ctrl = {
    'L': X_ctrl[:, valid_ctrl.index(l_col)].mean() if l_col in valid_ctrl else np.nan,
    'R': X_ctrl[:, valid_ctrl.index(r_col)].mean() if r_col in valid_ctrl else np.nan,
}
means_high = {
    'L': X_harm[:, valid_cols.index(l_col)].mean() if l_col in valid_cols else np.nan,
    'R': X_harm[:, valid_cols.index(r_col)].mean() if r_col in valid_cols else np.nan,
}

plot_brain_asymmetry_schematic(
    means_ctrl, means_high,
    save_path=fig_dir / "brain_asymmetry_schematic.png",
)
